In [1]:
# Extract capacity, capacity addition from ESM and aggregate by location

from zen_garden.postprocess.results import Results
import pandas as pd
from pathlib import Path

# Configuration
dataset_name = "2020_70ts_30a"
selected_technologies = {"heat_pump", "photovoltaics", "wind_offshore", "wind_onshore"}
specific_locations = {"DE", "CH", "IT", "CZ", "SE", "UK", "DK", "NL", "AT"}

base_path = Path("outputs")
results_path = base_path / dataset_name
output_dir = Path("csv_output") / dataset_name

output_dir.mkdir(parents=True, exist_ok=True)

# Load Results
res = Results(results_path)
capacity = res.get_full_ts("capacity").reset_index()
capacity_addition = res.get_full_ts("capacity_addition").reset_index()
capacity_previous = res.get_full_ts("capacity_previous").reset_index()

# Filter for selected technologies
capacity = capacity[capacity["technology"].isin(selected_technologies)]
capacity_addition = capacity_addition[capacity_addition["technology"].isin(selected_technologies)]
capacity_previous = capacity_previous[capacity_previous["technology"].isin(selected_technologies)]

capacity_addition.columns = capacity_addition.columns.map(str)
capacity_previous.columns = capacity_previous.columns.map(str)

year_cols = [col for col in capacity_addition.columns if col.isdigit()]
first_year_col = min(year_cols, key=int)

cap_prev_trimmed = capacity_previous[["technology", "capacity_type", "location", first_year_col]].copy()
cap_prev_trimmed = cap_prev_trimmed.rename(columns={first_year_col: "previous_capacity"})

capacity_addition = pd.merge(
    capacity_addition,
    cap_prev_trimmed,
    on=["technology", "capacity_type", "location"],
    how="left"
)

capacity_addition[first_year_col] = (
    capacity_addition[first_year_col] - capacity_addition["previous_capacity"].fillna(0)
).clip(lower=0)

capacity_addition = capacity_addition.drop(columns=["previous_capacity"])

# Save raw filtered outputs
capacity.round(4).to_csv(output_dir / "capacity.csv", index=False)
capacity_addition.round(4).to_csv(output_dir / "capacity_addition.csv", index=False)
capacity_previous.round(4).to_csv(output_dir / "capacity_previous.csv", index=False)

# Identify ROE (rest of Europe)
all_locations = set(capacity["location"].unique())
roe_locations = all_locations - specific_locations

# Aggregation helper
def aggregate_by_location(df, location_set, location_name):
    return (
        df[df["location"].isin(location_set)]
        .groupby(["technology", "capacity_type"])
        .sum(numeric_only=True)
        .assign(location=location_name)
    )

# Aggregation wrapper
def perform_aggregation(df):
    parts = [aggregate_by_location(df, {loc}, loc) for loc in specific_locations]
    parts.append(aggregate_by_location(df, roe_locations, "ROE"))
    return pd.concat(parts).reset_index()

# Aggregate
capacity_aggregated = perform_aggregation(capacity)
capacity_addition_aggregated = perform_aggregation(capacity_addition)

# Save aggregated outputs
capacity_aggregated.round(4).to_csv(output_dir / "capacity_aggregated_by_location.csv", index=False)
capacity_addition_aggregated.round(4).to_csv(output_dir / "capacity_addition_aggregated_by_location.csv", index=False)

print(f"All CSV files for dataset '{dataset_name}' saved in: {output_dir}")

All CSV files for dataset '2020_70ts_30a' saved in: csv_output\2020_70ts_30a


In [2]:
# rename_year_columns.py
"""
Renames numeric year columns (e.g. "0", "1", ...) in aggregated CSVs
to actual years (e.g. "2025", "2030", ...) using system.json config.
Also reorders columns so 'location' is the third column.
"""

import json
import pandas as pd
from pathlib import Path

# Configuration
dataset_name = "2020_70ts_30a"
results_path = Path("outputs") / dataset_name
output_dir = Path("csv_output") / dataset_name

# Load system.json and construct rename map
system_path = results_path / "system.json"
with open(system_path, "r") as f:
    config = json.load(f)

ref_year = config["reference_year"]
interval = config["interval_between_years"]
n_years = config["optimized_years"]

year_map = {str(i): str(ref_year + i * interval) for i in range(n_years)}

def rename_year_columns(csv_file):
    df = pd.read_csv(csv_file)
    df.rename(columns={col: year_map[col] for col in df.columns if col in year_map}, inplace=True)

    # Reorder columns to put location third
    base_cols = ["technology", "capacity_type", "location"]
    rest = [c for c in df.columns if c not in base_cols]
    df = df[base_cols + rest]

    df.to_csv(csv_file, index=False)
    print(f"Updated: {csv_file.name}")

# Apply to both aggregated files
rename_year_columns(output_dir / "capacity_aggregated_by_location.csv")
rename_year_columns(output_dir / "capacity_addition_aggregated_by_location.csv")

Updated: capacity_aggregated_by_location.csv
Updated: capacity_addition_aggregated_by_location.csv
